In [172]:
import pandas as pd
import numpy as np

In [173]:
df = pd.read_csv('swiggy_file.csv')
print('Raw dataset shape:', df.shape)
print('Columns:', df.columns.tolist())
df.head()

Raw dataset shape: (140657, 10)
Columns: ['Restaurant Name', 'Cuisine', 'Rating', 'Number of Ratings', 'Average Price', 'Number of Offers', 'Offer Name', 'Area', 'Pure Veg', 'Location']


,Restaurant Name,Cuisine,Rating,Number of Ratings,Average Price,Number of Offers,Offer Name,Area,Pure Veg,Location
0,La Pino'Z Pizza,"Pizzas, Pastas",4.0,10+ ratings,₹250 for two,2,FLAT DEAL\nFLAT ₹125 OFF\nUSE FLAT125ABOVE ₹69...,LALA LAJPAT RAI MARKET,No,Abohar
1,The Second Wife,"Indian, North Indian",3.6,50+ ratings,₹250 for two,2,"30% OFF UPTO ₹75\nUSE TRYNEWABOVE ₹149, FLAT ₹...",Central Abohar,No,Abohar
2,Tasty Bites,"Italian, Beverages",3.8,10+ ratings,₹200 for two,1,FLAT ₹120 OFF\nUSE AXIS120ABOVE ₹500,Central Abohar,Yes,Abohar
3,Food Studio,"Pizzas, Burgers",3.5,8 ratings,₹49 for two,5,"50% OFF UPTO ₹100\nUSE TRYNEWABOVE ₹129, FLAT ...",Central Abohar,Yes,Abohar
4,Roll Express,"Fast Food, Snacks",4.3,100+ ratings,₹200 for two,2,DEAL OF DAY\n10% OFF UPTO ₹40\nUSE STEALDEALAB...,Circular Road,No,Abohar


In [174]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
)

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

print('Columns:', df.columns.tolist())

Columns: ['restaurant_name', 'cuisine', 'rating', 'number_of_ratings', 'average_price', 'number_of_offers', 'offer_name', 'area', 'pure_veg', 'location']


In [175]:
before = len(df)
df = df.drop_duplicates()
print(f'Before: {before:,}')
print(f'After:  {len(df):,}')
print(f'Removed: {before - len(df):,} duplicates')

Before: 140,657
After:  139,321
Removed: 1,336 duplicates


In [176]:
metro_cities = ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad', 'Chennai', 'Kolkata']

# Match regardless of case
df = df[df['location'].str.title().isin(metro_cities)].copy()
df['location'] = df['location'].str.title()

print(f'Rows after metro filter: {len(df):,}')
print()
print('City-wise breakdown:')
print(df['location'].value_counts())

Rows after metro filter: 10,083

City-wise breakdown:
location
Hyderabad    1998
Kolkata      1974
Mumbai       1893
Delhi        1783
Bangalore    1720
Chennai       715
Name: count, dtype: int64


*Business Interpretation: We focus on India's 6 major metropolitan cities — Mumbai, Delhi, Bangalore, Hyderabad, Chennai, and Kolkata — which represent Swiggy's highest-density urban markets. This gives us 10,083 restaurant records, well above the 10,000 minimum, and allows meaningful city-to-city comparisons in our Tableau dashboard.*

In [177]:
df['rating_num'] = df['rating'].replace({'NEW': np.nan, '--': np.nan})
df['rating_num'] = pd.to_numeric(df['rating_num'], errors='coerce')

total = len(df)
nulls = df['rating_num'].isnull().sum()

print(f'Total restaurants:  {total:,}')
print(f'With valid rating:  {total - nulls:,} ({(total-nulls)/total*100:.1f}%)')
print(f'Without rating:     {nulls:,} ({nulls/total*100:.1f}%)')
print(f'  - NEW: {(df["rating"] == "NEW").sum():,}')
print(f'  - --:  {(df["rating"] == "--").sum():,}')
print(f'Rating range: {df["rating_num"].min()} to {df["rating_num"].max()}')

Total restaurants:  10,083
With valid rating:  7,738 (76.7%)
Without rating:     2,345 (23.3%)
  - NEW: 1,035
  - --:  1,310
Rating range: 1.5 to 5.0


In [178]:
df['rating'] = df['rating_num']
df = df.drop(columns=['rating_num'])

*Business Interpretation: After replacing 'NEW' and '--' with NaN, roughly 24% of metro restaurants lack a meaningful rating. These are newly launched or dormant listings. We exclude them from rating-based analysis but retain them for count and coverage metrics.*

In [179]:
df['price_num'] = (
    df['average_price']
    .str.replace('₹', '', regex=False)
    .str.replace('\u20b9', '', regex=False)
    .str.replace(' for two', '', regex=False)
    .str.strip()
)
df['price_num'] = pd.to_numeric(df['price_num'], errors='coerce')

print('=== Before outlier treatment ===')
print(f'Price range: ₹{df["price_num"].min():.0f} to ₹{df["price_num"].max():.0f}')
print(f'Prices ≤ ₹10:    {(df["price_num"] <= 10).sum()} rows')
print(f'Prices > ₹5000:  {(df["price_num"] > 5000).sum()} rows')

# Cap scraping errors
df.loc[df['price_num'] < 20, 'price_num'] = np.nan
df.loc[df['price_num'] > 5000, 'price_num'] = np.nan

print()
print('=== After outlier treatment ===')
print(f'Price range: ₹{df["price_num"].min():.0f} to ₹{df["price_num"].max():.0f}')
print(f'Price nulls: {df["price_num"].isnull().sum()}')

=== Before outlier treatment ===
Price range: ₹1 to ₹2000
Prices ≤ ₹10:    22 rows
Prices > ₹5000:  0 rows

=== After outlier treatment ===
Price range: ₹20 to ₹2000
Price nulls: 25


In [180]:
def parse_rating_count(val):
    if pd.isna(val):
        return np.nan
    val = str(val).strip()
    if val == 'Too Few Ratings':
        return np.nan
    val = val.replace('+ ratings', '').replace(' ratings', '').strip()
    if 'K' in val.upper():
        try:
            return int(float(val.upper().replace('K', '')) * 1000)
        except ValueError:
            return np.nan
    try:
        return int(float(val))
    except ValueError:
        return np.nan

df['ratings_count'] = df['number_of_ratings'].apply(parse_rating_count)

print(f'Valid counts: {df["ratings_count"].notna().sum():,}')
print(f'Null counts:  {df["ratings_count"].isnull().sum():,}')
print()
print(df['ratings_count'].describe().round(0))

Valid counts: 7,738
Null counts:  2,345

count     7738.0
mean       883.0
std       2289.0
min          2.0
25%         10.0
50%        100.0
75%        500.0
max      10000.0
Name: ratings_count, dtype: float64


In [181]:
# Primary cuisine = first item from comma-separated list
df['primary_cuisine'] = df['cuisine'].str.split(',').str[0].str.strip()
df['primary_cuisine'] = df['primary_cuisine'].fillna('Unknown')

# Binary veg flag
df['is_veg'] = df['pure_veg'].map({'Yes': 1, 'No': 0})

# Clean offers column
df['num_offers'] = df['number_of_offers'].astype(int)

print('=== Top 15 Cuisines ===')
print(df['primary_cuisine'].value_counts().head(15))
print()
print('=== Veg Split ===')
print(df['is_veg'].value_counts().rename({1: 'Veg', 0: 'Non-Veg'}))

=== Top 15 Cuisines ===
primary_cuisine
Indian          1292
Chinese         1224
North Indian    1130
Biryani          749
South Indian     707
Bakery           591
Beverages        557
Snacks           459
Fast Food        368
Pizzas           355
Desserts         347
Ice Cream        230
Burgers          189
Sweets           182
Italian          133
Name: count, dtype: int64

=== Veg Split ===
is_veg
Non-Veg    6522
Veg        3561
Name: count, dtype: int64


In [182]:
print('=' * 55)
print('CLEANING SUMMARY')
print('=' * 55)
print(f'Final shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()

print('── Null Counts ──')
for col in ['rating', 'price_num', 'ratings_count', 'primary_cuisine', 'is_veg', 'area', 'location']:
    nulls = df[col].isnull().sum()
    print(f'  {col:20s} {nulls:>5,} nulls')

print()
print('── City Breakdown ──')
print(df['location'].value_counts())

print()
assert len(df) > 10000, f'ERROR: Only {len(df)} rows!'
print(f'✓ {len(df):,} rows — exceeds 10,000 minimum')

CLEANING SUMMARY
Final shape: 10,083 rows × 15 columns

── Null Counts ──
  rating               2,345 nulls
  price_num               25 nulls
  ratings_count        2,345 nulls
  primary_cuisine          0 nulls
  is_veg                   0 nulls
  area                     0 nulls
  location                 0 nulls

── City Breakdown ──
location
Hyderabad    1998
Kolkata      1974
Mumbai       1893
Delhi        1783
Bangalore    1720
Chennai       715
Name: count, dtype: int64

✓ 10,083 rows — exceeds 10,000 minimum


In [183]:
# Fill remaining nulls for Tableau-friendly output
df['ratings_count'] = df['ratings_count'].fillna(0)
df['price_num'] = df['price_num'].fillna(df['price_num'].median())
df['rating'] = df['rating'].fillna(0)
df['offer_name'] = df['offer_name'].fillna('No Offer')
df['number_of_ratings'] = df['number_of_ratings'].fillna('No Ratings')

print('Null check after filling:')
print(df.isnull().sum())

Null check after filling:
restaurant_name      0
cuisine              0
rating               0
number_of_ratings    0
average_price        0
number_of_offers     0
offer_name           0
area                 0
pure_veg             0
location             0
price_num            0
ratings_count        0
primary_cuisine      0
is_veg               0
num_offers           0
dtype: int64


In [184]:
OUTPUT_PATH = 'cleaned_dataset.csv'
df.to_csv(OUTPUT_PATH, index=False)

print(f'Saved: {OUTPUT_PATH}')
print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
print()
print('Columns exported:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

Saved: cleaned_dataset.csv
Shape: 10,083 rows × 15 columns

Columns exported:
   1. restaurant_name
   2. cuisine
   3. rating
   4. number_of_ratings
   5. average_price
   6. number_of_offers
   7. offer_name
   8. area
   9. pure_veg
  10. location
  11. price_num
  12. ratings_count
  13. primary_cuisine
  14. is_veg
  15. num_offers
